In [3]:
from ml_enhance import QuantumFPDatasetBuilder, RDKitFeatureCalculator, get_preprocessed_smiles, canonicalize_smiles
from pathlib import Path
from rdkit import Chem
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
def build_solubility_lookup(raw_df: pd.DataFrame) -> pd.Series:
    """
    Returns a Series indexed by canonical SMILES containing solubility.
    """
    df = raw_df[["SMILES", "Solubility"]].copy()

    df["canon_smiles"] = df["SMILES"].apply(canonicalize_smiles)
    df = df.dropna(subset=["canon_smiles"])

    return df.set_index("canon_smiles")["Solubility"]

def attach_solubility(feature_df: pd.DataFrame, sol_lookup: pd.Series) -> pd.DataFrame:
    df = feature_df.copy()

    df["canon_smiles"] = df["smiles"].apply(canonicalize_smiles)
    df["solubility"] = df["canon_smiles"].map(sol_lookup)

    return df

In [5]:
DATA_SOURCE = Path("../data/AqSolDB/data_curated.csv")
raw_data = pd.read_csv(DATA_SOURCE)
# QFP_preprocessed_smiles = np.array([get_preprocessed_smiles(smiles) for smiles in raw_data["SMILES"]])
# Write the smiles to JSON
# Run data through QuantumFP

In [6]:
QFP_OUTPUT_PATH = Path("../data/QuantumFP/QFP_output")
builder = QuantumFPDatasetBuilder(QFP_OUTPUT_PATH)
df, errors = builder.build_dataset(multiprocess=True)

100%|██████████| 8837/8837 [38:18<00:00,  3.84it/s]  


In [10]:
errors

[]

In [7]:
rdkit_calculator = RDKitFeatureCalculator()
df = rdkit_calculator.add_to_dataframe(df, multiprocess=True)

100%|██████████| 8837/8837 [00:47<00:00, 187.70it/s]


In [8]:
# Add solutbility:
solubility_lookup = build_solubility_lookup(raw_data)
df = attach_solubility(df, solubility_lookup)

In [9]:
df.head()

,smiles,id,energy,atomization_energy,homo_lumo_gap,ionization_energy,electron_affinity,chemical_potential,molecular_dipole_norm,molecular_quadrupole_principal_invariant_2,molecular_quadrupole_principal_invariant_3,molecular_polarizability_mean,molecular_polarizability_anisotropy,enthalpy,zero_point_energy,radius_of_gyration,molecular_volume,sterimol_L,sterimol_Bmin,sterimol_Bmax,molecular_sasa,solvation_energy_water,solvation_energy_thf,solvation_energy_cyclohexane,solvation_energy_dmso,gibbs_free_energy_300K,entropy_300K,heat_capacity_300K,ir_mode_count_1500,ir_centroid_freq_1500,ir_norm_intensity_1500,ir_mode_count_1500_2750,ir_centroid_freq_1500_2750,ir_norm_intensity_1500_2750,ir_mode_count_2750_4000,ir_centroid_freq_2750_4000,ir_norm_intensity_2750_4000,avg_partial_charge_cyclohexane,avg_partial_charge_dmso,avg_partial_charge_thf,...,fr_imidazole,fr_imide,fr_isocyan,fr_isothiocyan,fr_ketone,fr_ketone_Topliss,fr_lactam,fr_lactone,fr_methoxy,fr_morpholine,fr_nitrile,fr_nitro,fr_nitro_arom,fr_nitro_arom_nonortho,fr_nitroso,fr_oxazole,fr_oxime,fr_para_hydroxylation,fr_phenol,fr_phenol_noOrthoHbond,fr_phos_acid,fr_phos_ester,fr_piperdine,fr_piperzine,fr_priamide,fr_prisulfonamd,fr_pyridine,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,canon_smiles,solubility
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,0,-34.129003,5.239994,0.089513,0.476438,0.201907,-0.338539,1.513984,-191.687655,-37.755113,-99.744189,14693.529251,58.559026,92.688029,2.186860,989.116670,9.059914,1.700001,5.254968,698.863882,-0.012962,-0.011137,-0.006143,-0.012831,30.822382,0.092455,0.009831,43.000000,900.202435,0.711247,4.000000,1743.932325,0.152361,7.000000,3237.962943,0.136392,1.665335e-17,-1.615375e-15,-9.825474e-16,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,O=C1Nc2cccc3cccc1c23,-3.254767
1,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,1,-26.123298,3.330826,0.105979,0.507545,0.200001,-0.356143,0.965182,-188.636399,720.869344,-95.248037,11823.272346,34.383902,60.507200,2.247185,782.057567,9.990874,1.750000,3.959554,611.482580,-0.008349,-0.007164,-0.003944,-0.008264,7.997456,0.087955,0.008734,28.000000,910.719228,0.727210,3.000000,1684.199429,0.135077,5.000000,3088.815542,0.137712,-2.299748e-16,-4.202987e-16,-9.992007e-16,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,O=Cc1ccc(Cl)cc1,-2.177078
2,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,10,-45.121773,7.140100,0.154440,0.463887,0.109782,-0.296101,0.909688,-169.958738,291.625442,-98.287723,6703.370243,129.313459,174.435232,3.135000,1406.870919,12.931461,2.427156,4.436073,958.980151,-0.011616,-0.010083,-0.005716,-0.011507,92.698411,0.122050,0.012387,69.000000,1042.467615,0.752676,6.000000,1516.461948,0.131871,18.000000,3037.991787,0.115452,1.342820e-16,1.897976e-16,-4.601364e-17,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,CC(C)(C)c1ccc(OCC2CO2)cc1,-3.430239
3,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,100,-39.453112,5.850939,0.073945,0.299042,0.004866,-0.145684,5.103982,-1306.769545,16565.487506,-90.301117,6006.343973,76.347615,115.800728,2.476355,1120.000242,10.226343,1.700006,4.793257,789.413162,-0.106643,-0.092824,-0.053028,-0.105662,43.537025,0.109369,0.010882,51.522146,1056.135534,0.816506,3.000004,1809.907500,0.073527,11.000022,2979.551717,0.109966,4.166573e-02,4.166573e-02,4.166573e-02,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Cc1cccc(C)c1OCC(=O)O,-2.256645
4,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,999,-72.427791,8.519924,0.068689,0.479849,0.249515,-0.373551,2.145071,-670.362641,-1839.250752,-230.053983,42083.450391,68.632785,141.060576,4.121926,1872.213947,16.721224,2.039677,6.125659,1293.442713,-0.029540,-0.025321,-0.013926,-0.029235,21.472017,0.157203,0.015074,79.000000,1033.796493,0.748754,7.000000,1713.210384,0.098557,10.000000,3133.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8837 entries, 0 to 8836
Columns: 280 entries, smiles to solubility
dtypes: float64(167), int64(111), str(2)
memory usage: 18.9 MB


In [ ]:
df_clean = df.dropna() # mostly consisting of the molecules with transition metals as they cause problems with the partial charge calculation of RDKit
df_clean.info()

<class 'pandas.DataFrame'>
Index: 8763 entries, 0 to 8836
Columns: 280 entries, smiles to solubility
dtypes: float64(167), int64(111), str(2)
memory usage: 18.8 MB


In [13]:
df_clean.to_csv("../data/processed_dataset_wo_metals.csv", index=False)

In [13]:
from collections import Counter

df_clean = df.dropna()

print()

total_atom_types = []
count = 0
for smiles in df_clean["canon_smiles"]:
    mol = Chem.MolFromSmiles(smiles)

    mol_atom_types = [atom.GetSymbol() for atom in mol.GetAtoms()]

    total_atom_types.extend(mol_atom_types)

print(Counter(total_atom_types))


Counter({'C': 104255, 'O': 21909, 'N': 10451, 'Cl': 3715, 'S': 1853, 'F': 1204, 'Br': 468, 'P': 332, 'I': 133, 'Si': 93, 'B': 7, 'Al': 1, 'H': 1})


In [14]:
Counter({'C': 104752, 'O': 22047, 'N': 10486, 'Cl': 3754, 'S': 1874, 'F': 1204, 'Br': 474, 'P': 332, 'I': 133, 'Si': 93, 'Sn': 18, 'Se': 10, 'B': 9, 'Pb': 6, 'Co': 5, 'Zn': 4, 'Cu': 4, 'As': 4, 'V': 3, 'Mo': 3, 'Cd': 3, 'Mn': 2, 'W': 2, 'Nb': 2, 'Cr': 2, 'Te': 2, 'Sb': 2, 'Ge': 2, 'Ca': 1, 'Bi': 1, 'Al': 1, 'Sr': 1, 'H': 1, 'Fe': 1, 'Ni': 1, 'Ti': 1, 'Zr': 1, 'Be': 1, 'Ce': 1, 'Hg': 1, 'Y': 1})

Counter({'C': 104752,
         'O': 22047,
         'N': 10486,
         'Cl': 3754,
         'S': 1874,
         'F': 1204,
         'Br': 474,
         'P': 332,
         'I': 133,
         'Si': 93,
         'Sn': 18,
         'Se': 10,
         'B': 9,
         'Pb': 6,
         'Co': 5,
         'Zn': 4,
         'Cu': 4,
         'As': 4,
         'V': 3,
         'Mo': 3,
         'Cd': 3,
         'Mn': 2,
         'W': 2,
         'Nb': 2,
         'Cr': 2,
         'Te': 2,
         'Sb': 2,
         'Ge': 2,
         'Ca': 1,
         'Bi': 1,
         'Al': 1,
         'Sr': 1,
         'H': 1,
         'Fe': 1,
         'Ni': 1,
         'Ti': 1,
         'Zr': 1,
         'Be': 1,
         'Ce': 1,
         'Hg': 1,
         'Y': 1})